<a href="https://colab.research.google.com/github/justorfc/Est_Python_R_2026_1/blob/main/11_Semana_11_Sesion_02_Aplicacion_Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Vamos directo a la acción. En esta sesión, los estudiantes aprenderán a cuantificar exactamente qué tan "peligroso" es un punto de datos anómalo para su modelo , utilizando herramientas avanzadas de diagnóstico. Es la diferencia entre sospechar que un dato está mal y demostrar estadísticamente que está arruinando las predicciones.

### 📘 Estructura del Notebook: Semana 11 - Sesión 2

**Título del Notebook:** `Semana_11_Sesion_02_Aplicacion_Diagnostico.ipynb`

#### **Celda 1: Markdown (Encabezado y Fase 1: IA)**

# SEMANA 11: Diagnóstico Avanzado del Modelo Lineal
**Asignatura:** Estadística Aplicada con Python y R
**Sesión 2:** Aplicación Práctica: Leverage, Distancia de Cook y Observaciones Influyentes.

## Fase 1: Actividad "Estudia y Aprende" (Aplicación)
Antes de ejecutar el código de diagnóstico, interactúa con tu IA de preferencia usando el siguiente prompt para entender qué significan estas métricas avanzadas y cómo se interpretan.

**PROMPT 2 - APLICACIÓN:**
> Actúa como tutor experto en diagnóstico de regresión con Python y R.
> 1. Muéstrame cómo graficar residuos.
> 2. Cómo evaluar homocedasticidad.
> 3. Cómo calcular leverage y distancia de Cook.
> 4. Cómo detectar observaciones influyentes.
> 5. Interpreta resultados como lo haría un ingeniero.
> No solo muestres código; explica qué significa cada resultado.

#### **Celda 2: Markdown (Fase 2: Demostración Docente - Contexto)**

## Fase 2: Demostración Docente en Python
Anteriormente vimos gráficamente cómo un punto anómalo distorsionaba el modelo. Hoy lo mediremos numéricamente usando `statsmodels.graphics`.

Conceptos clave para hoy:
* **Leverage (Apalancamiento):** Mide qué tan extremo es un punto en el eje X. Un punto muy alejado del resto tiene un alto "potencial" de jalar la línea.
* **Distancia de Cook:** Mide cuánto cambiarían TODOS los coeficientes del modelo si decidiéramos eliminar ese punto específico. Si la distancia es mayor a 1 (o destaca abruptamente del resto), el punto es peligrosamente influyente.

**Pregunta clave:** ¿Este modelo es confiable para tomar decisiones reales?

#### **Celda 3: Código Python (Cálculo de Cook y Gráfico de Influencia)**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
np.random.seed(100)

# Recreamos el dataset "enfermo" de la sesión anterior (Prensa vs Deformación)
presion_X = np.linspace(10, 100, 50)
error_creciente = np.random.normal(0, presion_X * 0.15, 50)
deformacion_Y = 5.0 + 2.5 * presion_X + error_creciente

# Introducimos el punto influyente (Índice 49)
presion_X[49] = 150
deformacion_Y[49] = 100

df_prensa = pd.DataFrame({'Presion': presion_X, 'Deformacion': deformacion_Y})

# Ajustamos el modelo
X_const = sm.add_constant(df_prensa['Presion'])
modelo_enfermo = sm.OLS(df_prensa['Deformacion'], X_const).fit()

# ==========================================
# DIAGNÓSTICO AVANZADO: INFLUENCIA Y COOK
# ==========================================
# Extraemos las medidas de influencia usando OLSInfluence
influencia = modelo_enfermo.get_influence()
distancia_cook = influencia.cooks_distance[0]

# Graficamos la Distancia de Cook
plt.figure(figsize=(10, 5))
plt.stem(np.arange(len(distancia_cook)), distancia_cook, markerfmt=",")
plt.title('Distancia de Cook por Observación')
plt.xlabel('Índice de la Observación')
plt.ylabel('Distancia de Cook')
plt.axhline(y=4/len(df_prensa), color='r', linestyle='--', label='Umbral Crítico sugerido (4/n)')
plt.legend()
plt.show()

# Interpretación Ingenieril:
# Fíjate en el último punto (índice 49). Su Distancia de Cook rompe el gráfico.
# Esto significa que este único dato está definiendo el modelo casi por sí solo.
# En ingeniería, si construyes basándote en este modelo, estás tomando decisiones
# basadas en un posible error de medición (un "outlier"), no en el comportamiento real del material.

#### **Celda 4: Código Python (Influence Plot Integrado)**

In [ ]:
# ==========================================
# EL GRÁFICO DE INFLUENCIA (Influence Plot)
# ==========================================
# Este gráfico combina todo: Residuos (eje Y), Leverage (eje X) y Cook (tamaño del círculo)

fig, ax = plt.subplots(figsize=(10, 6))
fig = sm.graphics.influence_plot(modelo_enfermo, ax=ax, criterion="cooks")
plt.title('Gráfico de Influencia (Leverage vs Residuos)')
plt.show()

# Interpretación:
# - Eje X (Leverage): El punto 49 está muy a la derecha, indicando que la presión de 150 es extrema respecto al resto.
# - Eje Y (Residuos estudentizados): Qué tan mal predijo el modelo ese punto.
# - Tamaño del círculo: Es gigante para el punto 49, confirmando que tiene una Distancia de Cook masiva.
# Conclusión: Este modelo NO es confiable. Debemos investigar el punto 49, corregirlo si es un error tipográfico,
# o eliminarlo y justificar técnicamente por qué lo hicimos.

#### **Celda 5: Markdown (Transición a R y RMarkdown)**

## Trabajo Autónomo: Transición a R y RMarkdown

R tiene comandos increíblemente eficientes para extraer toda esta información numérica de golpe utilizando funciones como `influence.measures()`.

**Paso 1: Validación en Colab (Entorno R)**
1. Abre un nuevo Notebook en Colab y cambia el entorno a **R**.
2. Ejecuta el siguiente código para evaluar numéricamente el leverage y la distancia de Cook.

```R
# Recrear el dataset "enfermo"
set.seed(100)
presion <- seq(10, 100, length.out=50)
error_het <- rnorm(50, mean=0, sd=presion*0.15)
deformacion <- 5.0 + 2.5 * presion + error_het
presion[50] <- 150
deformacion[50] <- 100
datos_r <- data.frame(Presion=presion, Deformacion=deformacion)

# Ajustar modelo
modelo_r <- lm(Deformacion ~ Presion, data=datos_r)

# 1. Extraer la distancia de Cook de cada punto
cooks_d <- cooks.distance(modelo_r)
print("Distancia de Cook de los últimos 5 puntos:")
print(round(tail(cooks_d, 5), 4)) # Verás cómo el punto 50 es masivamente distinto

# 2. Resumen completo de medidas de influencia
# Esta función marca con un asterisco (*) las observaciones que superan umbrales críticos
medidas_influencia <- influence.measures(modelo_r)
print("Medidas de influencia para la observación anómala (Punto 50):")
print(medidas_influencia$infmat[50, ])

```

**Paso 2: Documentación en Posit Cloud y RPubs**

1. En **Posit Cloud**, crea tu documento **RMarkdown**.
2. Transfiere el código de R.
3. Escribe tu análisis respondiendo a la gran pregunta: *¿Por qué un punto con alto "Leverage" (apalancamiento) no siempre es malo, pero un punto con alta "Distancia de Cook" siempre es motivo de alerta roja para un ingeniero?*
4. Compila a HTML y publica en **RPubs**.

#### **Celda 6: Markdown (CIERRE DEL TEMA Y HOJA EVALUABLE)**

## CIERRE DEL TEMA: Generación de Resumen Guía

Hemos terminado el bloque de diagnóstico avanzado. Nunca más volverás a ver un $R^2$ alto sin sospechar. Consolida tus conocimientos preparando tu hoja evaluable.

**PROMPT DE CIERRE GLOBAL:**
> Genera un resumen estructurado para escribir a mano en UNA sola hoja.
> Formato obligatorio:
> A) Idea central (1-2 líneas).
> B) 6-10 viñetas organizadas lógicamente.
> C) 3 relaciones clave (por qué/cómo).
> D) 1 ejemplo aplicado a ingeniería.
> E) 3 preguntas de autoevaluación con respuesta.
> F) Cierre: "Hoy aprendí que ..."

⚠️ **RECUERDA (EVIDENCIA EVALUABLE):**
En el reverso de tu hoja física titulada *"Diagnóstico Avanzado del Modelo Lineal"*, debes escribir a mano:
* La idea central sobre la necesidad de diagnosticar un modelo.
* Qué son los residuos y cómo deben comportarse (patrón aleatorio).
* La diferencia entre Homocedasticidad y Heterocedasticidad.
* Qué significan el Leverage y la Distancia de Cook.
* Las consecuencias prácticas de violar los supuestos en ingeniería.
* Un ejemplo aplicado.
* Las 3 preguntas con sus respuestas.
* Tu reflexión personal ("Hoy aprendí que...").
